In [49]:
import pandas as pd
import os
import requests
from functools import reduce # tool for merging multiple dataframes together easily
from tqdm import tqdm # library for showing progress bars

# Create output directory if it doesn't exist
os.makedirs("data/raw", exist_ok=True)

In [50]:
def get_asylum_data(cities, decision, represented, col_name, nationwide=False):
    cases_dfs = []
    if nationwide:
        cities = {'Nationwide': 'All'}
    for city in tqdm(cities.values()):
        url = f'https://tracreports.org/phptools/immigration/asylum/graph.php?stat=count&timescale=fymon&base_city={str(city)}&represented={str(represented)}&denial2={str(decision)}&timeunit=number'
        response = requests.get(url)
        data = response.json()
        df = pd.DataFrame(data['timeline'])
        df['city'] = [k for k, v in cities.items() if v == city][0]
        df.rename(columns={'number': col_name}, inplace=True)
        cases_dfs.append(df)
    cases = pd.concat(cases_dfs)
    return cases

In [51]:
# Create cities dict. Each city has a numeric code in the API. Philly's is 30 (base_city = 30 parameter below)
# To get the data for the whole country, base_city=All is used, reflected in the function above
cities = {'Philadelphia': 30}

In [52]:
# Get all cases. Ensure "All" is passed with a captial A for representation and decision parameters
all_cases = get_asylum_data(cities, decision='All', col_name='all_cases', represented='All')

# Get denied cases. The code for Denied is 1
denied_cases = get_asylum_data(cities, decision=1, col_name='denied_cases', represented='All')

# Get all cases with representation. The code for represented is 2
all_represented = get_asylum_data(cities, decision='All', col_name='all_represented', represented=2)

# Get all cases without representation. The code for not represented is 1
all_not_represented = get_asylum_data(cities, decision='All', col_name='all_not_represented', represented=1)

# Get all cases with representation that were denied
represented_denied = get_asylum_data(cities, decision=1, col_name='represented_denied', represented=2)

# Get all cases without representation that were denied
not_represented_denied = get_asylum_data(cities, decision=1, col_name='not_represented_denied', represented=1)

100%|██████████| 1/1 [00:00<00:00, 14.77it/s]


In [53]:
# Merge all the dataframes together
dfs = [all_cases, denied_cases, all_represented, all_not_represented, represented_denied, not_represented_denied]
for df in dfs: 
    df.drop(columns=['percent'], inplace=True) # drop pct column which we will recalculate
    if df is not all_cases: # keep city column in one of the dfs, otherwise will repeat
        df.drop(columns=['city'], inplace=True)

asylum_data = reduce(lambda left, right: pd.merge(left, right, on="fymon"), dfs)

In [54]:
asylum_data.to_csv("data/raw/asylum_cases_philadelphia.csv", index=False)